# Train U-Net on miniature dataset
This notebook shows a minimal training loop using the `UNet2Dto3D` model and
HDF5 dataset generated by the simple simulator. It also includes utilities to
inspect data shapes, run a short training loop, and visualise example
reconstructions.

## Cell 1 — Set project root and verify dataset file

In [ ]:
import os
from pathlib import Path

# Walk up from CWD until we find pyproject.toml (= project root)
candidate = Path.cwd()
for _ in range(10):
    if (candidate / 'pyproject.toml').exists():
        break
    candidate = candidate.parent

os.chdir(candidate)
PROJECT_ROOT = Path.cwd()
print('Working directory:', PROJECT_ROOT)

H5_PATH = PROJECT_ROOT / 'data' / 'raw' / 'simple_mini_100s.h5'
print('Dataset :', H5_PATH)
print('Exists  :', H5_PATH.exists())

if not H5_PATH.exists():
    raise FileNotFoundError(
        f'Dataset not found: {H5_PATH}\n'
        'Generate it first with:\n'
        '  python scripts/run_simulation.py --config configs/simulations/simple_mini.yaml --n_samples 100'
    )

## Cell 2 — Inspect HDF5 metadata

In [ ]:
import h5py

with h5py.File(H5_PATH, 'r') as f:
    print('Top-level keys:', list(f.keys()))
    if 'metadata' in f:
        print('\nMetadata attributes:')
        for k, v in f['metadata'].attrs.items():
            print(f'  {k}: {v}')
    if 'samples' in f:
        sample_keys = sorted(f['samples'].keys())
        print(f'\nNumber of samples: {len(sample_keys)}')
        grp0 = f['samples'][sample_keys[0]]
        print('Sample 0 datasets:')
        for k in grp0:
            print(f'  {k}: shape={grp0[k].shape}  dtype={grp0[k].dtype}')

## Cell 3 — Imports

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
import numpy as np
import matplotlib.pyplot as plt
from spectangle.simulations.io import load_simulation
from spectangle.models.unet import UNet2Dto3D

print('Imports OK')

## Cell 4 — Dataset class and instantiation

In [ ]:
class H5MiniDataset(Dataset):
    def __init__(self, path):
        data = load_simulation(str(path))
        self.samples = data['samples']
        meta = data['metadata']

        # pad_y / pad_x may be absent in files generated by older code.
        # Derive them from the shapes stored in the first sample.
        if 'pad_y' in meta and 'pad_x' in meta:
            self.pad_y = int(float(meta['pad_y']))
            self.pad_x = int(float(meta['pad_x']))
        else:
            # Infer from spectrogram vs cube spatial dimensions
            s0 = self.samples[0]
            spec_shape = s0['spectrograms'].shape   # (4, ny+2*pad_y, nx+2*pad_x)
            cube_shape  = s0['cube'].shape           # (n_lambda, ny, nx)
            self.pad_y = (spec_shape[1] - cube_shape[1]) // 2
            self.pad_x = (spec_shape[2] - cube_shape[2]) // 2
            print(f'[INFO] pad_y/pad_x not in metadata — inferred: pad_y={self.pad_y}, pad_x={self.pad_x}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        spec = s['spectrograms'].astype(np.float32)
        cube = s['cube'].astype(np.float32)
        direct = s.get('direct_image')
        if direct is not None:
            x = np.concatenate([spec, direct[None, ...]], axis=0)
        else:
            x = spec
        return torch.from_numpy(x), torch.from_numpy(cube)


# Use the absolute path resolved in Cell 1
ds = H5MiniDataset(H5_PATH)
print(f'Dataset loaded: {len(ds)} samples  pad_x={ds.pad_x}  pad_y={ds.pad_y}')

## Cell 5 — DataLoader + model

In [ ]:
loader = DataLoader(ds, batch_size=4, shuffle=True)

# Infer in_channels from first sample (4 spectrograms, +1 if direct image present)
x0, y0 = ds[0]
in_channels = x0.shape[0]
n_lambda    = y0.shape[0]
scene_ny    = y0.shape[1]
scene_nx    = y0.shape[2]
print(f'in_channels={in_channels}  n_lambda={n_lambda}  scene={scene_ny}x{scene_nx}')

model = UNet2Dto3D(
    in_channels=in_channels,
    n_lambda=n_lambda,
    base_features=16,
    depth=3,
    scene_shape=(scene_ny, scene_nx),
)
opt = torch.optim.Adam(model.parameters(), lr=1e-4)

x_batch, y_batch = next(iter(loader))
print('x batch shape:', x_batch.shape)
print('y batch shape:', y_batch.shape)

## Cell 6 — Quick data visualisation

In [ ]:
spec0  = x_batch[0, 0].numpy()
broad0 = y_batch[0].numpy().sum(axis=0)

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(spec0,  origin='lower', cmap='inferno')
ax[0].set_title('Spectrogram ch0 (batch 0)')
ax[0].axis('off')
ax[1].imshow(broad0, origin='lower', cmap='gray')
ax[1].set_title('Ground-truth broadband (batch 0)')
ax[1].axis('off')
plt.tight_layout()
plt.show()

## Cell 7 — Training loop (3 epochs)

In [ ]:
loss_history = []
for epoch in range(3):
    for x, y in loader:
        pred = model(x)
        loss = torch.nn.functional.mse_loss(pred, y)
        opt.zero_grad()
        loss.backward()
        opt.step()
    loss_history.append(loss.item())
    print(f'Epoch {epoch}  loss={loss.item():.6f}')

plt.figure(figsize=(5, 3))
plt.plot(loss_history, marker='o')
plt.xlabel('Epoch')
plt.ylabel('MSE loss')
plt.title('Training loss (toy run)')
plt.grid(True)
plt.tight_layout()
plt.show()

## Cell 8 — Reconstruction visualisation

In [ ]:
model.eval()
with torch.no_grad():
    x0t, y0t = ds[0]
    pred0 = model(x0t.unsqueeze(0)).squeeze(0).cpu().numpy()  # (n_lambda, ny, nx)

gt_broad   = y0t.numpy().sum(axis=0)
pred_broad = pred0.sum(axis=0)

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(gt_broad,                           origin='lower', cmap='gray')
ax[0].set_title('GT broadband');   ax[0].axis('off')
ax[1].imshow(pred_broad,                         origin='lower', cmap='gray')
ax[1].set_title('Pred broadband'); ax[1].axis('off')
ax[2].imshow(np.abs(pred_broad - gt_broad),      origin='lower', cmap='magma')
ax[2].set_title('Absolute residual'); ax[2].axis('off')
plt.tight_layout()
plt.show()

If you'd like, I can convert the toy training loop to a fully featured
training script (with checkpointing, LR scheduler and mixed-precision), and
add a callback to visualise reconstructions at the end of each epoch.